# 03 Dodatkowe analizy – walidacja modelu emocji i sentymentu

## 1. Konfiguracja

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

from src.config import EMOTIONS, LABELS_DIR, PHASE_ORDER, POLITICAL_SUBREDDITS, PROCESSED_DIR, RANDOM_STATE, RESULTS_DIR

VALIDATION_DIR = PROCESSED_DIR / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_CYCLES = [2016, 2024]
PER_CELL = 10

## 2. Wybór 400 komentarzy do ręcznego oznaczenia

In [ ]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "camp", "cycle", "phase", "text"])
political = sample[sample["subreddit"].isin(POLITICAL_SUBREDDITS) & sample["cycle"].isin(VALIDATION_CYCLES)
                   & sample["phase"].isin(PHASE_ORDER)]
chosen = political.sample(frac=1, random_state=RANDOM_STATE).groupby(["cycle", "subreddit", "phase"]).head(PER_CELL)
print(len(chosen))
chosen.groupby(["cycle", "subreddit", "phase"]).size().unstack("phase")

In [ ]:
chosen[["id", "text"]].sample(frac=1, random_state=RANDOM_STATE).to_csv(VALIDATION_DIR / "emotion_to_label.csv", index=False)

## 3. Etykiety ręczne (`labels/emotion_labels.csv`: `id`, `label`)

In [ ]:
labels = pd.read_csv(LABELS_DIR / "emotion_labels.csv", dtype={"id": str})
labels["label"].value_counts()

## 4. Porównanie z emocją o najwyższym prawdopodobieństwie

In [ ]:
emotions = pd.concat(pd.read_parquet(path) for path in sorted((PROCESSED_DIR / "emotions").glob("*.parquet")))
predicted = emotions.set_index("id")[EMOTIONS].idxmax(axis=1).rename("predicted").reset_index()
data = chosen[["id", "camp", "cycle"]].merge(labels, on="id").merge(predicted, on="id")
print(len(data))

In [ ]:
scores = []
for name, subset in [("dem", data[data["camp"] == "dem"]), ("rep", data[data["camp"] == "rep"]), ("razem", data)]:
    scores.append({"grupa": name, "n": len(subset), "f1_macro": f1_score(subset["label"], subset["predicted"], average="macro")})
scores = pd.DataFrame(scores)
scores.to_csv(RESULTS_DIR / "validation_emotions_f1.csv", index=False)
scores

In [ ]:
for name, subset in [("dem", data[data["camp"] == "dem"]), ("rep", data[data["camp"] == "rep"])]:
    matrix = pd.DataFrame(confusion_matrix(subset["label"], subset["predicted"], labels=EMOTIONS), index=EMOTIONS, columns=EMOTIONS)
    matrix.to_csv(RESULTS_DIR / f"validation_emotions_confusion_{name}.csv")
    print(name)
    print(matrix)

## 5. Zgodność z modelem sentymentu (korelacja poziomów tygodniowych)

In [ ]:
weekly = pd.read_csv(PROCESSED_DIR / "weekly_emotions.csv")
by_group = weekly.groupby(["subreddit", "cycle"])
agreement = pd.DataFrame({"r_negative": by_group.apply(lambda group: group["negative"].corr(group["sent_neg_share"])),
                          "r_joy": by_group.apply(lambda group: group["joy"].corr(group["sent_pos_share"]))})
total = pd.DataFrame({"r_negative": [weekly["negative"].corr(weekly["sent_neg_share"])],
                      "r_joy": [weekly["joy"].corr(weekly["sent_pos_share"])]}, index=["razem"])
agreement = pd.concat([agreement, total])
agreement.to_csv(RESULTS_DIR / "validation_sentiment_corr.csv")
agreement.round(3)